# Mapa de Movilidad - ITT (Ciudad Paraiso / Pulmon de Oriente)

Mapa interactivo de la dimension **Movilidad** del Indice de Transformacion
Territorial (ITT). Fuente oficial: `data/1_data_itt/` (capas 06 y 07).

- **Siniestros viales** (`06_movilidad_siniestros.geojson`, 97 puntos): puntos por
  severidad (lesionados / mortal / solo danos) y mapa de calor.
- **Red peatonal** (`07_movilidad_red_peatonal_osm.geojson`, 400 lineas, OSM): lineas
  por tipo de via.

Estandares del proyecto: paleta Okabe-Ito (apta daltonismo), capas base Google
(sin OpenStreetMap), mascara oscura fuera de la zona, panel colapsable con serie
anual (Chart.js + fallback a tabla), slider de intensidad del calor y boton unico
'Exportar lamina (deck)'. Ejecutar en el entorno uv unico del proyecto.


## 1. Entorno y paleta Okabe-Ito

In [ ]:
# Dependencias: ejecutar en el entorno uv UNICO del proyecto.
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
# Paleta Okabe-Ito (accesible para daltonismo) - misma paleta que el resto del proyecto
OKI_AZUL     = "#0072B2"
OKI_NARANJA  = "#E69F00"
OKI_VERDE    = "#009E73"
OKI_AMARILLO = "#F0E442"
OKI_AZUL_C   = "#56B4E9"
OKI_BERMELL  = "#D55E00"
OKI_MORADO   = "#CC79A7"
OKI_GRIS     = "#333333"
BG = "#F4F6F9"
CRS_WEB    = 4326   # WGS84, capa web
CRS_METRIC = 9377   # MAGNA-SIRGAS Origen Nacional, calculo metrico
print("Entorno listo. Paleta Okabe-Ito cargada.")

## 2. Rutas y poligono de la zona

In [ ]:
# Resolucion robusta del poligono de Ciudad Paraiso (local uv + Colab).
def _resolver_poligono():
    env = os.environ.get("POLIGONO_ZONA")
    cand = [env] if env else []
    cand += [
        os.path.join("..", "data", "geojson", "poligono_ciudad_Paraiso.geojson"),
        os.path.join("data", "geojson", "poligono_ciudad_Paraiso.geojson"),
        "/content/data/geojson/poligono_ciudad_Paraiso.geojson",
        "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/poligono_ciudad_Paraiso.geojson",
    ]
    for p in cand:
        if p and os.path.isfile(p):
            return os.path.abspath(p)
    return None
_poly_path = _resolver_poligono()
POLIGONO_ZONA = None
if _poly_path:
    POLIGONO_ZONA = gpd.read_file(_poly_path).to_crs(CRS_WEB)
    print("Poligono de zona:", _poly_path, "| features:", len(POLIGONO_ZONA))
else:
    print("Aviso: no se encontro poligono_ciudad_Paraiso.geojson; el mapa se dibuja sin contorno de zona.")
# Carpeta de salida procesada (gitignored).
OUT_DIR = None
for _base in [os.path.join("..", "data_processed_geo"), "data_processed_geo",
              "/content/data_processed_geo"]:
    try:
        os.makedirs(_base, exist_ok=True)
        OUT_DIR = os.path.abspath(_base)
        break
    except Exception:
        continue
print("Salida procesada:", OUT_DIR)
def _ruta_itt(nombre):
    cand = [
        os.path.join("..", "data", "1_data_itt", nombre),
        os.path.join("data", "1_data_itt", nombre),
        os.path.join("/content", "data", "1_data_itt", nombre),
        os.path.join("/content", "drive", "MyDrive",
                     "mapa_Analitico_ciudad_Paraiso", "data", "1_data_itt", nombre),
    ]
    for p in cand:
        if os.path.isfile(p):
            return os.path.abspath(p)
    return None

## 3. Carga de datos reales de movilidad (data/1_data_itt)

In [ ]:
# 3a. Siniestros viales (puntos) y red peatonal (lineas).
_rp_sin = _ruta_itt("06_movilidad_siniestros.geojson")
if _rp_sin is None:
    raise FileNotFoundError("No se encontro 06_movilidad_siniestros.geojson en data/1_data_itt.")
sin = gpd.read_file(_rp_sin).to_crs(CRS_WEB)
sin = sin[sin.geometry.notna()].copy()
sin["fecha"] = pd.to_datetime(sin["fecha_evento"], errors="coerce")
sin["anio"] = sin["fecha"].dt.year
sin["mes"] = sin["fecha"].dt.strftime("%Y-%m")
# Severidad normalizada a 3 clases.
_SEV_MAP = {"lesionados": "lesionados", "mortal": "mortal", "solo_danos": "solo_danos"}
sin["severidad"] = (sin["clasificacion_severidad"].astype(str).str.strip().str.lower()
                    .map(_SEV_MAP).fillna("lesionados"))
# Clase de accidente legible (unificar mayus/minus).
sin["clase"] = (sin["clase_acci"].astype(str).str.strip().str.upper()
                .replace({"NAN": "SIN DATO", "NONE": "SIN DATO", "": "SIN DATO"}))
sin["lon"] = sin.geometry.x
sin["lat"] = sin.geometry.y
SEV_CATS = ["lesionados", "mortal", "solo_danos"]
print("Siniestros:", len(sin))
print(sin["severidad"].value_counts().reindex(SEV_CATS).to_string())
print("Por anio:", sin["anio"].value_counts().sort_index().to_dict())
# Red peatonal (OSM).
_rp_red = _ruta_itt("07_movilidad_red_peatonal_osm.geojson")
if _rp_red is None:
    raise FileNotFoundError("No se encontro 07_movilidad_red_peatonal_osm.geojson en data/1_data_itt.")
red = gpd.read_file(_rp_red).to_crs(CRS_WEB)
red = red[red.geometry.notna()].copy()
red["highway"] = red["highway"].astype(str)
# Longitud total aproximada (m) de la red peatonal en la zona (CRS metrico).
_red_m = red.to_crs(CRS_METRIC)
RED_LONG_M = float(_red_m.geometry.length.sum())
print("Red peatonal: lineas =", len(red), "| longitud total =", round(RED_LONG_M), "m")
# Serie mensual de siniestros (para el panel), por severidad.
serie_mensual = (sin.dropna(subset=["fecha"])
                    .assign(periodo=lambda d: d["fecha"].dt.strftime("%Y-%m"))
                    .groupby(["periodo", "severidad"]).size()
                    .reset_index(name="conteo"))
print("Serie mensual filas:", len(serie_mensual))

## 4. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #mov-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #mov-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #mov-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 5. Construccion del mapa interactivo (Folium)

In [ ]:
# 5a. Mapa Folium de movilidad: puntos de siniestros por severidad + heatmap + red peatonal.
import folium
from folium.plugins import HeatMap
# Color de punto por severidad (Okabe-Ito, accesible). Mortal = bermellon (alerta).
SEV_PUNTO = {"lesionados": OKI_NARANJA, "mortal": OKI_BERMELL, "solo_danos": OKI_AZUL_C}
SEV_BORDE = {"lesionados": "#8a5a00", "mortal": "#7a2f00", "solo_danos": "#1f6f9c"}
SEV_RADIO = {"lesionados": 4, "mortal": 6, "solo_danos": 3}
SEV_LABEL = {"lesionados": "lesionados", "mortal": "mortales", "solo_danos": "solo danos"}
# Gradiente de calor de siniestros (rampa secuencial naranja ColorBrewer, apta daltonismo).
GRAD_SIN = {0.0: "#feedde", 0.4: "#fdbe85", 0.7: "#fd8d3c", 1.0: "#a63603"}  # Oranges
# Color por tipo de via peatonal (de peatonal puro a vehicular). Okabe-Ito.
VIA_COLOR = {
    "footway":     OKI_VERDE,     # andenes / senderos peatonales
    "pedestrian":  OKI_AZUL,      # calle peatonal
    "residential": OKI_NARANJA,   # via residencial
    "service":     OKI_AMARILLO,  # via de servicio
    "tertiary":    OKI_AZUL_C,    # terciaria
    "secondary":   OKI_MORADO,    # secundaria
    "primary":     OKI_BERMELL,   # primaria
}
def _color_via(h):
    return VIA_COLOR.get(str(h).strip().lower(), "#999999")
# Centro del mapa.
if POLIGONO_ZONA is not None:
    _c = POLIGONO_ZONA.geometry.union_all().centroid
    centro = [_c.y, _c.x]
else:
    centro = [sin["lat"].mean(), sin["lon"].mean()]
m_mov = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True,
                   zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m_mov.get_root().width = "100%"
m_mov.get_root().height = "100%"
_css_full = ("<style>html,body{height:100%;margin:0;padding:0}"
             ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;"
             "width:100%;height:100%}</style>")
m_mov.get_root().header.add_child(folium.Element(_css_full))
# Capas base conmutables (Google Streets activa). Sin OpenStreetMap.
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m_mov)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_mov)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_mov)
# Mascara oscura fuera de la zona (foco).
if POLIGONO_ZONA is not None:
    _zona_geom = POLIGONO_ZONA.geometry.union_all()
    _mundo = [[-180.0, -85.0], [180.0, -85.0], [180.0, 85.0], [-180.0, 85.0], [-180.0, -85.0]]
    _agujeros = []
    _geoms = list(_zona_geom.geoms) if _zona_geom.geom_type == "MultiPolygon" else [_zona_geom]
    for _g in _geoms:
        _agujeros.append([[x, y] for x, y in _g.exterior.coords])
    _mascara = {"type": "Feature", "properties": {},
                "geometry": {"type": "Polygon", "coordinates": [_mundo] + _agujeros}}
    folium.GeoJson(_mascara, name="Mascara (foco en la zona)",
        style_function=lambda f: {"fillColor": "#000000", "color": "#000000",
                                  "weight": 0, "fillOpacity": 0.25},
        control=False).add_to(m_mov)
# Limite oficial de la zona (borde dorado punteado). Activo.
_fg_limite = None
_rp_lim = _ruta_itt("00_limite_ciudad_paraiso.geojson")
if _rp_lim is not None:
    _lim = gpd.read_file(_rp_lim).to_crs(CRS_WEB)
    _fg_limite = folium.FeatureGroup(name="Limite Ciudad Paraiso", show=True)
    folium.GeoJson(_lim.to_json(),
        style_function=lambda f: {"color": "#8A6D1A", "weight": 5, "dashArray": "10 6", "fill": False}
    ).add_to(_fg_limite)
# Puntos solo dentro de la zona.
if POLIGONO_ZONA is not None:
    _zona = POLIGONO_ZONA.geometry.union_all()
    sin_z = sin[sin.geometry.within(_zona) | sin.geometry.intersects(_zona)].copy()
else:
    sin_z = sin.copy()
# Capas de PUNTOS: solo siniestros con fecha >= 2025-09-01 (no afecta calor ni graficas).
_FECHA_CORTE_PUNTOS = pd.Timestamp("2025-09-01")
sin_z_pts = sin_z[pd.to_datetime(sin_z["fecha"], errors="coerce") >= _FECHA_CORTE_PUNTOS].copy()
# Heatmap de siniestros (TODOS los eventos en zona, sin filtrar fecha).
_pts_calor = sin_z[["lat", "lon"]].dropna().values.tolist()
_fg_calor = folium.FeatureGroup(name="Calor: siniestros (" + str(len(_pts_calor)) + ")", show=False)
if _pts_calor:
    HeatMap(_pts_calor, gradient=GRAD_SIN, radius=18, blur=14, min_opacity=0.35,
            name="heat_siniestros").add_to(_fg_calor)
# Capas de puntos por severidad (CircleMarker, popup SIN PII). Apagadas al inicio.
def _popup_sin(r):
    f = r["fecha"].strftime("%Y-%m-%d") if pd.notna(r["fecha"]) else "sin fecha"
    clase = str(r.get("clase") or "SIN DATO").capitalize()
    sev = SEV_LABEL.get(r["severidad"], r["severidad"])
    filas = '<div style="font-weight:700;color:%s">Siniestro vial - %s</div>' % (SEV_PUNTO[r["severidad"]], sev)
    filas += '<div>Fecha: %s</div>' % f
    filas += '<div>Clase: %s</div>' % clase
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">%s</div>'
            % (BG, filas))
_fg_puntos = {}
for sev in SEV_CATS:
    sub = sin_z_pts[sin_z_pts["severidad"] == sev]
    fg = folium.FeatureGroup(name="Puntos: " + SEV_LABEL[sev] + " (" + str(len(sub)) + ")", show=(sev == "mortal"))
    for _, r in sub.iterrows():
        if pd.isna(r["lat"]) or pd.isna(r["lon"]):
            continue
        folium.CircleMarker(location=[r["lat"], r["lon"]], radius=SEV_RADIO[sev],
            color=SEV_BORDE[sev], weight=1, fill=True,
            fill_color=SEV_PUNTO[sev], fill_opacity=0.8,
            popup=folium.Popup(_popup_sin(r), max_width=240)).add_to(fg)
    _fg_puntos[sev] = fg
# Red peatonal (lineas) por tipo de via.
if POLIGONO_ZONA is not None:
    red_z = red[red.geometry.intersects(POLIGONO_ZONA.geometry.union_all())].copy()
else:
    red_z = red.copy()
_fg_red = folium.FeatureGroup(name="Red peatonal (OSM, " + str(len(red_z)) + " vias)", show=False)
for _, r in red_z.iterrows():
    _col = _color_via(r["highway"])
    try:
        _lng = round(float(r.get("length") or 0), 1)
    except Exception:
        _lng = 0
    _nm = str(r.get("name") or "").strip()
    _pop = ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:150px">'
            '<div style="font-weight:700;color:%s">Via: %s</div>'
            '%s<div>Longitud: %s m</div></div>'
            % (BG, _col, str(r["highway"]),
               ('<div>Nombre: %s</div>' % _nm) if _nm and _nm.lower() != "nan" else "",
               _lng))
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f, c=_col: {"color": c, "weight": 2.5, "opacity": 0.85}),
        tooltip="Via: %s" % str(r["highway"]),
        popup=folium.Popup(_pop, max_width=220)).add_to(_fg_red)
# --- Orden de overlays en el control de capas ---
_orden = [
    _fg_limite,
    _fg_red,
    _fg_puntos.get("mortal"),
    _fg_puntos.get("lesionados"),
    _fg_puntos.get("solo_danos"),
    _fg_calor,
]
for _capa in _orden:
    if _capa is not None:
        _capa.add_to(m_mov)
if POLIGONO_ZONA is not None:
    m_mov.fit_bounds([[3.442445, -76.53259], [3.44791, -76.526004]], padding=(30, 30))
folium.LayerControl(collapsed=False, position="topright").add_to(m_mov)
print("Mapa de movilidad construido. Siniestros en zona:", len(sin_z), "| vias en zona:", len(red_z))

## 6. Panel colapsable: resumen + serie de tiempo

In [ ]:
# 6a. Panel flotante colapsable (resumen + serie anual Chart.js con fallback a tabla).
_serie_anual = serie_mensual.copy()
_serie_anual["anio"] = _serie_anual["periodo"].astype(str).str[:4]
_periodos = sorted(_serie_anual["anio"].dropna().unique().tolist())
_pivot = (_serie_anual.pivot_table(index="anio", columns="severidad", values="conteo", aggfunc="sum")
          .reindex(_periodos).fillna(0).astype(int))
for _c in SEV_CATS:
    if _c not in _pivot.columns:
        _pivot[_c] = 0
_series_cat = {c: _pivot[c].tolist() for c in SEV_CATS}
_LINEA = {"lesionados": OKI_NARANJA, "mortal": OKI_BERMELL, "solo_danos": OKI_AZUL_C}
_LAB = {"lesionados": "Lesionados", "mortal": "Mortales", "solo_danos": "Solo danos"}
_tot_cat = sin["severidad"].value_counts().to_dict()
_tot_total = int(len(sin))
_resumen_items = (
    '<div style="margin:2px 0 4px 0">Siniestros (total): <b>%d</b> &nbsp;|&nbsp; '
    'Red peatonal: <b>%s km</b></div>' % (_tot_total, ("%.1f" % (RED_LONG_M / 1000.0)))
)
_resumen_items += "".join(
    '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:2px;border:1px solid #777"></i>'
    '<span>%s: <b>%d</b></span></div>'
    % (_LINEA[c], _LAB[c], int(_tot_cat.get(c, 0))) for c in SEV_CATS
)
_filas_tabla = ""
for p in _periodos:
    celdas = "".join('<td style="padding:1px 5px;text-align:right">%d</td>' % int(_pivot.loc[p, c]) for c in SEV_CATS)
    _filas_tabla += '<tr><td style="padding:1px 5px">%s</td>%s</tr>' % (p, celdas)
_cab_tabla = "".join('<th style="padding:1px 5px">%s</th>' % _LAB[c] for c in SEV_CATS)
_tabla_html = (
    '<table style="font-size:10px;border-collapse:collapse;width:100%%">'
    '<tr style="border-bottom:1px solid #ccc"><th style="padding:1px 5px">Ano</th>%s</tr>%s</table>'
    % (_cab_tabla, _filas_tabla))
# Eje unico (magnitudes similares, lesionados domina pero sin orden de 10x).
_chart_data = json.dumps({
    "labels": _periodos,
    "series": [{"label": _LAB[c], "data": _series_cat[c], "color": _LINEA[c], "axis": "y"} for c in SEV_CATS],
}, ensure_ascii=False)
_panel = """
<div id="mov-panel" style="position:fixed;top:20px;left:20px;z-index:9999;
     background:__BG__;max-width:480px;width:min(480px,46vw);
     padding:10px 12px;border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="font-weight:700;margin-bottom:4px">Movilidad - resumen</div>
  __RESUMEN__
  <details style="margin-top:6px">
    <summary style="cursor:pointer;font-weight:600;color:__AZUL__;outline:none">
      Ver serie de tiempo (anual) / Ocultar</summary>
    <div style="margin-top:8px">
      <div style="font-size:9px;color:#666;margin-bottom:4px">Siniestros viales por severidad y ano.</div>
      <div style="position:relative;height:200px;width:100%"><canvas id="mov-chart"></canvas></div>
      <div id="mov-fallback" style="display:none;max-height:160px;overflow:auto">__TABLA__</div>
    </div>
  </details>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var DATA = __DATA__;
  function dibujar(){
    var cv = document.getElementById('mov-chart');
    var fb = document.getElementById('mov-fallback');
    if (typeof Chart === 'undefined' || !cv){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
      return;
    }
    try {
      new Chart(cv.getContext('2d'), {
        type: 'line',
        data: {labels: DATA.labels,
          datasets: DATA.series.map(function(s){
            return {label: s.label, data: s.data, borderColor: s.color,
                    backgroundColor: s.color, borderWidth: 2, pointRadius: 2,
                    tension: 0.25, fill: false, yAxisID: (s.axis||'y')};
          })},
        options: {responsive:true, maintainAspectRatio:false, interaction:{mode:'index',intersect:false},
          plugins:{legend:{labels:{boxWidth:10,font:{size:10}}}},
          scales:{x:{ticks:{font:{size:9}}},
                  y:{type:'linear',position:'left',beginAtZero:true,
                     title:{display:true,text:'Cantidad de siniestros',font:{size:9}},
                     ticks:{font:{size:9}}}}}
      });
    } catch(e){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
    }
  }
  if (document.readyState === 'complete'){ setTimeout(dibujar, 300); }
  else { window.addEventListener('load', function(){ setTimeout(dibujar, 300); }); }
})();
</script>
"""
_panel = (_panel.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS)
                .replace("__AZUL__", OKI_AZUL).replace("__RESUMEN__", _resumen_items)
                .replace("__TABLA__", _tabla_html).replace("__DATA__", _chart_data))
m_mov.get_root().html.add_child(folium.Element(_panel))
print("Panel colapsable inyectado (resumen + serie de tiempo).")
# Slider de intensidad del heatmap (runtime, no cambia la base).
_MAP_ID = m_mov.get_name()
_slider_html = """
<div id="heat-intensity-ctrl" style="position:fixed;bottom:24px;right:24px;z-index:9999;
     background:__BG__;padding:8px 12px;border-radius:10px;
     box-shadow:0 2px 10px rgba(0,0,0,.3);font-family:system-ui,Segoe UI,sans-serif;
     font-size:12px;color:__GRIS__;max-width:220px">
  <div style="font-weight:700;margin-bottom:4px">Intensidad del calor</div>
  <input id="heat-intensity-range" type="range" min="8" max="40" value="18"
         style="width:100%;cursor:pointer">
  <div style="font-size:10px;color:#666;margin-top:2px">
    Radio: <span id="heat-intensity-val">18</span> px</div>
</div>
<script>
(function(){
  function aplicar(valor){
    var mp = window["__MAP_ID__"];
    if (!mp || typeof mp.eachLayer !== "function"){ return; }
    var r = parseInt(valor, 10);
    var b = Math.round(r * 0.8);
    mp.eachLayer(function(l){
      if (l && typeof l.setOptions === "function" && l._latlngs){
        try { l.setOptions({radius: r, blur: b}); if (typeof l.redraw === "function"){ l.redraw(); } } catch(e){}
      }
    });
    var vs = document.getElementById("heat-intensity-val");
    if (vs){ vs.textContent = r; }
  }
  function init(){
    var inp = document.getElementById("heat-intensity-range");
    if (!inp){ return; }
    inp.addEventListener("input", function(){ aplicar(this.value); });
  }
  if (document.readyState === "complete"){ setTimeout(init, 400); }
  else { window.addEventListener("load", function(){ setTimeout(init, 400); }); }
})();
</script>
"""
_slider_html = (_slider_html.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS).replace("__MAP_ID__", _MAP_ID))
m_mov.get_root().html.add_child(folium.Element(_slider_html))
print("Slider de intensidad del heatmap inyectado.")

## 7. Exportacion del HTML

In [ ]:
# 7a. Exportar el mapa de movilidad a data_processed_geo/.
agregar_export(m_mov, "Movilidad", "movilidad")
html_mov = os.path.join(OUT_DIR, "mapa_movilidad.html")
m_mov.save(html_mov)
print("Mapa de movilidad exportado:", html_mov)
m_mov